# AI4Pain 2026 — ARM vs HAND deep-learning training

Runs `scripts/19_armhand_dl.py` from the GitHub repo against the private Kaggle dataset `soyebjim/ai4pain-2026-data`.

**Before first run, add a Kaggle Secret** (`Add-ons → Secrets`) named `GITHUB_TOKEN` with a fine-scoped GitHub PAT that has `repo:read` on `soyeb-jim285/ai4pain-2026-analysis`.

Outputs (`/kaggle/working/results`, `/kaggle/working/plots`) are kept by Kaggle as the notebook's saved version.

In [ ]:
# --- Cell 1: install missing deps + clone repo ----------------------------
import os, sys, subprocess, glob

MISSING = ['neurokit2', 'umap-learn', 'pingouin', 'pywavelets']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *MISSING], check=True)

from kaggle_secrets import UserSecretsClient
_secrets = UserSecretsClient()
_token = _secrets.get_secret('GITHUB_TOKEN')

REPO_DIR = '/kaggle/working/repo'
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ['git', 'clone',
         f'https://{_token}@github.com/soyeb-jim285/ai4pain-2026-analysis.git',
         REPO_DIR],
        check=True,
    )
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Auto-discover dataset root. Kaggle's mount path may include
# /kaggle/input/datasets/<user>/<slug>/ instead of /kaggle/input/<slug>/,
# so glob for the canonical train/Bvp/ directory.
matches = glob.glob('/kaggle/input/**/train/Bvp', recursive=True)
if not matches:
    inp_root = '/kaggle/input'
    contents = sorted(os.listdir(inp_root)) if os.path.isdir(inp_root) else []
    raise SystemExit(f'No train/Bvp/ found under {inp_root}. /kaggle/input = {contents}')
DATA_ROOT = matches[0][: -len('/train/Bvp')]

os.environ['AI4PAIN_ROOT'] = DATA_ROOT
os.environ['AI4PAIN_CACHE'] = '/kaggle/working/cache'
os.environ['AI4PAIN_OUTPUT_DIR'] = '/kaggle/working'

print('repo head:', subprocess.run(
    ['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
    capture_output=True, text=True).stdout.strip())
print('AI4PAIN_ROOT  =', os.environ['AI4PAIN_ROOT'])
print('cwd           =', os.getcwd())
for _split in ('train', 'validation'):
    _p = os.path.join(os.environ['AI4PAIN_ROOT'], _split, 'Bvp')
    _n = len(os.listdir(_p)) if os.path.isdir(_p) else 'MISSING'
    print(f'  {_split:<10} -> {_p} ({_n} csvs)')

In [ ]:
# --- Cell 2: GPU sanity check + cache prime ------------------------------
import torch
print('torch          :', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('n_gpus         :', torch.cuda.device_count())

# Build / load cached numpy + parquet tensors (writes to AI4PAIN_CACHE)
subprocess.run([sys.executable, 'scripts/00_prime_cache.py'], check=True)

In [ ]:
# --- Cell 3: run feature extraction + sweep + DL training (resumable) ----
# Order:
#   1. tierC paper-derived features (script 21) — RESP wavelet, arc length, fuzzy entropy
#   2. classical ML sweep (script 18) — auto-merges tierB + tierC; tight_pool variant
#   3. DL training (script 19) — supervised + SSL CNN, 5-seed ensemble per fold

# Wipe stale ML sweep state so the new tight_pool / tierC features take effect
for f in [
    'results/tables/armhand_search_perfold.csv',
    'results/tables/armhand_search_summary.csv',
    'results/tables/armhand_search_validation.csv',
    'results/tables/armhand_search_ensembles.csv',
    'results/tables/armhand_search_top.csv',
    'results/tables/armhand_search_feature_importance_top30.csv',
]:
    p = os.path.join(REPO_DIR, f)
    if os.path.exists(p):
        os.remove(p)
        print(f'removed stale {f}')

print('\n=== STEP 1: tierC paper features ===')
subprocess.run([sys.executable, 'scripts/21_paper_features.py'], check=True)

print('\n=== STEP 2: classical ML sweep ===')
subprocess.run([sys.executable, 'scripts/18_armhand_search.py'], check=True)

print('\n=== STEP 3: DL training (supervised + SSL, 5-seed ensemble) ===')
subprocess.run([sys.executable, 'scripts/19_armhand_dl.py', '--n-seeds', '5'], check=True)


In [ ]:
# --- Cell 4: copy results + plots to /kaggle/working so Kaggle persists them
import shutil
for sub in ('results', 'plots', 'cache'):
    src = os.path.join(REPO_DIR, sub)
    dst = os.path.join('/kaggle/working', sub)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'copied {src} -> {dst}')

# Print headline numbers
import pandas as pd
summary_fp = '/kaggle/working/results/tables/armhand_dl_summary.csv'
if os.path.exists(summary_fp):
    df = pd.read_csv(summary_fp)
    print('\n=== ARM vs HAND DL summary ===')
    print(df.to_string(index=False))